# IndicSent-M: Exploratory Data Analysis of Memotion 3
**Course**: CSET 346 - Natural Language Processing  
**Instructor**: Dr. Shubham Choudhary  
**Team**: Anya Yadav, Pragya Singh, Priyanshu Thakur, Ronit Jaiswal, Sparshman Gupta  

This notebook explores the **Memotion 3** dataset, a multimodal benchmark comprising code-mixed **Hinglish** (Hindi + English) memes labeled for sentiment analysis (Positive, Neutral, Negative).


## 1. Setup & Environment
We import standard scientific computing, data analysis, and visualization libraries.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import re

# Paths to processed dataset splits
DATA_DIR = '../data/processed'
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'Train Samples: {len(train_df):,}')
print(f'Validation Samples: {len(val_df):,}')
print(f'Test Samples: {len(test_df):,}')


## 2. Dataset Schema & First Look
Let's inspect the columns and sample data rows.


In [ ]:
train_df.head(3)


## 3. Sentiment Class Distribution
Understanding class balance across Train, Validation, and Test splits.


In [ ]:
classes = ['Positive', 'Neutral', 'Negative']
colors = ['#10B981', '#3B82F6', '#EF4444']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
splits = [('Train Set', train_df), ('Val Set', val_df), ('Test Set', test_df)]

for ax, (title, df) in zip(axes, splits):
    counts = [df['sentiment'].value_counts().get(c, 0) for c in classes]
    bars = ax.bar(classes, counts, color=colors, edgecolor='#1E293B', linewidth=1.2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Memes')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for bar in bars:
        h = bar.get_height()
        p = (h / len(df)) * 100
        ax.annotate(f'{h}\n({p:.1f}%)', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Visualizing Sample Multimodal Memes
Here we view 6 sample memes along with their cleaned OCR text and target sentiment.


In [ ]:
sample_rows = train_df.sample(6, random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    # Handle relative paths from notebook directory
    img_path = os.path.join('..', row['image_path'])
    try:
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
    except Exception as e:
        ax.text(0.5, 0.5, 'Image load error', ha='center', va='center')
    
    text_preview = row['text'][:55] + '...' if len(row['text']) > 55 else row['text']
    sent = row['sentiment']
    color = '#10B981' if sent == 'Positive' else ('#3B82F6' if sent == 'Neutral' else '#EF4444')
    ax.set_title(f'Sentiment: {sent}\n"{text_preview}"', fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('IndicSent-M Sample Memes from Memotion 3', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 5. Text Statistics & Code-Mixing Analysis
Memes often contain concise captions with code-mixed Hindi (Devanagari and Latin romanizations) and English.


In [ ]:
train_df['word_count'] = train_df['text'].apply(lambda x: len(str(x).split()))
train_df['char_count'] = train_df['text'].apply(lambda x: len(str(x)))

# Count Devanagari script presence (Unicode range U+0900 to U+097F)
devanagari_pattern = re.compile(r'[ऀ-ॿ]')
train_df['has_devanagari'] = train_df['text'].apply(lambda x: bool(devanagari_pattern.search(str(x))))

print('Word Count Statistics:')
print(train_df['word_count'].describe())

print(f'\nPercentage of memes with Devanagari script: {train_df["has_devanagari"].mean() * 100:.2f}%')
print(f'Percentage of memes in Romanized/English script: {(1 - train_df["has_devanagari"].mean()) * 100:.2f}%')


In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(train_df['word_count'], bins=30, color='#6366F1', edgecolor='#1E293B', alpha=0.85)
plt.title('Distribution of Meme Text Word Counts', fontsize=12, fontweight='bold')
plt.xlabel('Number of Words per Meme')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 6. Summary of Key Exploratory Insights for Faculty Review
1. **Multilingual Code-Mixing**: While English vocabulary is pervasive, syntax and slang are predominantly Hindi transliterated into Roman script (e.g. *'bhai'*, *'kya baat hai'*), motivating our use of multilingual **XLM-RoBERTa** over standard monolingual BERT.
2. **Class Imbalance**: Neutral memes form the plurality (~42.5%), followed by Positive (~32.6%) and Negative (~25.0%). We utilize **Class-Weighted Cross Entropy Loss** to ensure minority classes are not penalized.
3. **Visual Complementarity**: Standalone text is frequently ambiguous or humorous without visual cues, proving that multimodal fusion is required to capture contextual intent.
